In [3]:
from openai import OpenAI

client = OpenAI()
MODEL = "gpt-6-luna"

In [4]:
agent = client.beta.agents.create(
    model=MODEL,
    instructions="You speak short sentences. Like caveman.",
    tools=[
        {
            "type": "computer_use",
            "include_screenshots": True,
        }
    ],
)

In [ ]:
from IPython.display import Image, display

with client.beta.agents.sessions.create(
    agent_id=agent.id,
    environment={
        "type": "openai_hosted",
        "desktop": {
            "enabled": True,
        },
        "network": {"access": "enabled"},
    },
    input="Go to nomadcoders.co and show tell me what is in the frontpage that looks blue.",
    stream=True,
) as events:
    for event in events:
        if event.type == "agent.session.created":
            session_id = event.session.id
        elif event.type == "agent.session.turn.output_text.done":
            print(event.text)
        elif (
            event.type == "agent.session.turn.item.done"
            and event.item.type == "computer_use_call"
        ):
            print("Computer: ", event.item.title, event.item.status)
            if event.item.output:
                display(Image(url=event.item.output.image_url))
        else:
            print(event.type)